In [1]:
import io, sys
from dataclasses import replace
from pathlib import Path

repo = Path.cwd().parent
sys.path.insert(0, str(repo / "src"))
from cs4406_sem_project.harness import recovery as R

# RQ4 as configs/recovery_policies.yaml sets it out, priced from notebook 07's single repetitions.
EST = repo / "results/raw/grid-feasibility-20261111T064956Z/estimates.jsonl"
cfg = R.load_config(R.DEFAULT_CONFIG)
runs = R.plan(cfg)
table = R.load_estimates(EST)
print(f"memory cap {cfg['memory_cap_gib']} GiB, max_db_bytes {cfg['max_db_bytes']}")
R.dry_run(runs, EST)

memory cap 14 GiB, max_db_bytes {'S1': 134217728, 'S2': 134217728, 'S3': 134217728, 'S4': 1048576}
356 Runs, 219556 adapter repetitions (reps + warmup per Run, summed)
kind     arm       runs  adapter reps
trials   S0-go       16          1616
trials   S0-cpp      16          1616
trials   S1          24          2424
trials   S2          24          2424
trials   S3           8           808
trials   M0          16          1616
trials   M1          32          3232
trials   total      136         13736
offline  S0-go        4           404
offline  S0-cpp       4           404
offline  S1           4           404
offline  S2           4           404
offline  total       16          1616
sweep    S0-go       24         24024
sweep    S0-cpp      24         24024
sweep    S1          36         36036
sweep    S2          36         36036
sweep    S3          12         12012
sweep    M0          24         24024
sweep    M1          48         48048
sweep    total      204        204

In [2]:
# What each alternative to the spec's repetition counts and focus cells would cost, in expected hours.
# Runs are reshaped with dataclasses.replace and priced with the driver's own expected_run_s. "no cell" names
# the arms with no Run to price: S4 has no focus cell, and S1, S2 and S3 have no 1 GiB cell (notebook 08).
def hours(rs):
    return sum(R.expected_run_s(r, table) or 0 for r in rs) / 3600

def fewer_re_preprocess(rs, reps=30, min_f=0.5):
    """re_preprocess Runs whose servers keep cheating (trials, and the sweep at f >= min_f) at `reps` + warm-up."""
    keep_rare = lambda r: r.kind == "sweep" and r.fault_prob < min_f
    return [replace(r, reps=reps) if r.policy == "re_preprocess" and not keep_rare(r) else r for r in rs]

MiB = 1 << 20
shrunk = R.plan(cfg, db_sizes=[1024 * MiB], record_sizes=[1024])
priced = lambda rs: [r for r in rs if R.expected_run_s(r, table) is not None]
alternatives = {
    "spec as written": runs,
    "without re_preprocess": [r for r in runs if r.policy != "re_preprocess"],
    "re_preprocess at 30 + 1 reps in trials and at f >= 0.5": fewer_re_preprocess(runs),
    "same, and re_preprocess only at 32 B records": [r for r in fewer_re_preprocess(runs)
                                                       if r.policy != "re_preprocess" or r.record_bytes == 32],
    "spec's shrink: focus cells 1 GiB x 1 KiB only": shrunk,
}
for label, rs in alternatives.items():
    arms = sorted({r.arm for r in priced(rs)})
    gone = sorted(set(cfg["policies"]) - set(arms))
    print(f"{label:58s} {hours(rs):8.1f} h  ({hours(rs) / 24:5.1f} days)  measured: {', '.join(arms)}"
          + (f"; no cell: {', '.join(gone)}" if gone else ""))

spec as written                                              1115.5 h  ( 46.5 days)  measured: M0, M1, S0-cpp, S0-go, S1, S2, S3; no cell: S4
without re_preprocess                                          86.8 h  (  3.6 days)  measured: M0, M1, S0-cpp, S0-go, S1, S2, S3; no cell: S4
re_preprocess at 30 + 1 reps in trials and at f >= 0.5        209.9 h  (  8.7 days)  measured: M0, M1, S0-cpp, S0-go, S1, S2, S3; no cell: S4
same, and re_preprocess only at 32 B records                  143.1 h  (  6.0 days)  measured: M0, M1, S0-cpp, S0-go, S1, S2, S3; no cell: S4
spec's shrink: focus cells 1 GiB x 1 KiB only                  13.6 h  (  0.6 days)  measured: M0, M1, S0-cpp, S0-go; no cell: S1, S2, S3, S4
